# 🏥 Logistic Regression Application: Breast Cancer Diagnosis
## TTK4260: Multivariate Data Analysis & Machine Learning

**Instructor:** Adil Rasheed — Department of Engineering Cybernetics, NTNU  
**Semester:** Spring 2026

---

### 📖 Overview

In this notebook, we apply **logistic regression** to a real-world medical diagnostic problem:  
classifying breast tumour biopsies as **malignant** or **benign** from digitized cell-nucleus measurements.

We use the [**Breast Cancer Wisconsin (Diagnostic)**](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)) dataset — one of the most widely used benchmark datasets in medical machine learning.

### 🎯 Learning Objectives

1. ✅ Apply the **end-to-end classification pipeline** on real clinical data
2. ✅ Perform meaningful **exploratory data analysis** to understand feature distributions
3. ✅ Train, tune, and evaluate logistic regression models using **cross-entropy, ROC/AUC, precision/recall**
4. ✅ Compare **no regularization vs L1 vs L2** on a high-dimensional medical dataset
5. ✅ Interpret model coefficients as **clinical risk factors**
6. ✅ Understand precision–recall trade-offs in **medical decision making**
7. ✅ Visualize the data using **PCA-projected decision boundaries**

---

### 📋 Table of Contents

1. [Setup & Imports](#setup)
2. [Dataset Description & Loading](#data)
3. [Exploratory Data Analysis](#eda)
4. [Data Preprocessing](#preprocessing)
5. [Baseline Logistic Regression](#baseline)
6. [Model Evaluation](#evaluation)
7. [Threshold Tuning for Clinical Use](#threshold)
8. [Regularization Comparison](#regularization)
9. [Feature Importance & Clinical Interpretation](#features)
10. [PCA-Projected Decision Boundary](#pca)
11. [Summary & Clinical Insights](#summary)

<a id='setup'></a>
## 1. 🔧 Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.special import expit as sigmoid

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

BLUE = '#1f77b4'
RED = '#d62728'
ORANGE = '#ff7f0e'
GREEN = '#2ca02c'
PURPLE = '#9467bd'

SEED = 42
np.random.seed(SEED)

print('✅ All libraries loaded successfully!')

<a id='data'></a>
## 2. 📁 Dataset Description & Loading

### About the Dataset

The **Breast Cancer Wisconsin (Diagnostic)** dataset contains measurements computed from a digitized **Fine Needle Aspirate (FNA)** image of a breast mass. Each row represents one patient's biopsy.

| Property | Value |
|----------|-------|
| **Samples** | 569 patients |
| **Features** | 30 real-valued measurements |
| **Classes** | 2 — Malignant (cancer) vs. Benign (not cancer) |
| **Class balance** | 212 malignant (37%), 357 benign (63%) |
| **Source** | University of Wisconsin, Dr. William H. Wolberg |

For each cell nucleus, **10 base features** are computed, and for each the **mean**, **standard error**, and **worst** (mean of 3 largest values) are recorded, giving $10 \times 3 = 30$ features:

| # | Feature | Description |
|---|---------|-------------|
| 1 | Radius | Mean distance from center to boundary points |
| 2 | Texture | Std. dev. of gray-scale values |
| 3 | Perimeter | Perimeter of the nucleus |
| 4 | Area | Area of the nucleus |
| 5 | Smoothness | Local variation in radius lengths |
| 6 | Compactness | $\text{perimeter}^2 / \text{area} - 1$ |
| 7 | Concavity | Severity of concave portions of the contour |
| 8 | Concave points | Number of concave portions |
| 9 | Symmetry | Symmetry of the nucleus |
| 10 | Fractal dimension | "Coastline approximation" — boundary complexity |

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='diagnosis')

# Note: sklearn encodes Malignant=0, Benign=1
# We keep this convention: positive class (1) = Benign, negative class (0) = Malignant
target_names = data.target_names  # ['malignant', 'benign']

print(f'Dataset shape: {X.shape[0]} samples × {X.shape[1]} features')
print(f'Classes: {dict(zip(target_names, np.bincount(y)))}')
print(f'\nFirst 5 rows:')
X.head()

<a id='eda'></a>
## 3. 🔍 Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar chart
counts = np.bincount(y)
bars = axes[0].bar(target_names, counts, color=[RED, BLUE], edgecolor='k', linewidth=0.5)
for bar, c in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{c} ({c/len(y):.1%})', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Class Distribution', fontsize=14)

# Descriptive statistics
desc = X.describe().T[['mean', 'std', 'min', 'max']]
axes[1].axis('off')
axes[1].set_title('Feature Scale Summary', fontsize=14)
# Show min/max range across features
axes[1].barh(range(10), desc['mean'].values[:10], xerr=desc['std'].values[:10],
             color=BLUE, alpha=0.7, edgecolor='k', linewidth=0.3, capsize=3)
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([n.replace(' ', '\n') for n in data.feature_names[:10]], fontsize=8)
axes[1].set_xlabel('Mean (± std)', fontsize=10)
axes[1].set_title('First 10 Features: Scale Varies Widely!', fontsize=13)
axes[1].axvline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

print('⚠️  Observation: Features have very different scales (e.g., area ~ 100s vs smoothness ~ 0.1).')
print('   → Standardization is essential before fitting logistic regression!')

In [ ]:
# Distribution of key features by diagnosis
key_features = ['mean radius', 'mean texture', 'mean concavity', 'mean concave points',
                'worst radius', 'worst concavity']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feat in zip(axes.ravel(), key_features):
    for label, color, name in zip([0, 1], [RED, BLUE], target_names):
        ax.hist(X.loc[y == label, feat], bins=25, alpha=0.55, color=color,
                label=name, density=True, edgecolor='k', linewidth=0.3)
    ax.set_xlabel(feat, fontsize=11)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_title(feat.replace('mean ', '').replace('worst ', 'worst ').title(), fontsize=12)

plt.suptitle('Feature Distributions by Diagnosis — Malignant vs. Benign',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('📌 Insight: Malignant tumours tend to have larger radius, higher concavity, and more concave points.')
print('   These features show good class separation — promising for logistic regression.')

In [ ]:
# Correlation heatmap of the 10 mean features
mean_features = [f for f in X.columns if f.startswith('mean')]
corr = X[mean_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, vmin=-1, vmax=1, linewidths=0.5,
            xticklabels=[f.replace('mean ', '') for f in mean_features],
            yticklabels=[f.replace('mean ', '') for f in mean_features])
ax.set_title('Correlation Matrix — Mean Features', fontsize=14)
plt.tight_layout()
plt.show()

print('📌 Observation: radius, perimeter, and area are highly correlated (r > 0.95).')
print('   Compactness, concavity, and concave points are also correlated.')
print('   → Multicollinearity present — regularization (L2) will help stabilize the model.')

In [ ]:
# Pairwise scatter plot of the 4 most discriminating features
top_features = ['mean concave points', 'worst radius', 'worst concavity', 'mean radius']
df_plot = X[top_features].copy()
df_plot['Diagnosis'] = y.map({0: 'Malignant', 1: 'Benign'})

g = sns.pairplot(df_plot, hue='Diagnosis', palette={'Malignant': RED, 'Benign': BLUE},
                 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20, 'edgecolor': 'k', 'linewidth': 0.2},
                 height=2.5)
g.figure.suptitle('Pairwise Scatter of Top Discriminating Features', fontsize=14,
                  fontweight='bold', y=1.02)
plt.show()

print('📌 Classes are largely separable in these feature pairs — logistic regression should perform well.')

<a id='preprocessing'></a>
## 4. ⚙️ Data Preprocessing

Before fitting logistic regression, we:
1. **Split** into training (70%) and test (30%) sets with stratification
2. **Standardize** features (zero mean, unit variance) — fit on train, transform both

In [ ]:
# Train-test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples ({y_train.value_counts()[0]} malignant, {y_train.value_counts()[1]} benign)')
print(f'Test set:     {X_test.shape[0]} samples ({y_test.value_counts()[0]} malignant, {y_test.value_counts()[1]} benign)')

# Standardize features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on training data only
X_test_sc = scaler.transform(X_test)          # transform test data with training statistics

print(f'\n✅ Features standardized.')
print(f'   Training mean ≈ {X_train_sc.mean(axis=0).mean():.4f}, std ≈ {X_train_sc.std(axis=0).mean():.4f}')
print(f'   Test mean ≈ {X_test_sc.mean(axis=0).mean():.4f} (slight deviation from 0 is expected)')

<a id='baseline'></a>
## 5. 📐 Baseline Logistic Regression

We start with a standard L2-regularized logistic regression (default in scikit-learn).

Recall the model:

$$
P(\text{benign} \mid \boldsymbol{x}) = \sigma(\boldsymbol{\theta}^\top\boldsymbol{x}+\theta_0) = \frac{1}{1+e^{-(\boldsymbol{\theta}^\top\boldsymbol{x}+\theta_0)}}
$$

In [ ]:
# Fit baseline model
clf_base = LogisticRegression(C=1.0, penalty='l2', max_iter=5000, random_state=SEED)
clf_base.fit(X_train_sc, y_train)

# Predictions
y_pred = clf_base.predict(X_test_sc)
y_prob = clf_base.predict_proba(X_test_sc)[:, 1]  # P(benign)

# Cross-validation score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(clf_base, X_train_sc, y_train, cv=cv, scoring='roc_auc')

print('╔════════════════════════════════════════╗')
print('║     BASELINE MODEL RESULTS             ║')
print('╠════════════════════════════════════════╣')
print(f'║  Test Accuracy:    {accuracy_score(y_test, y_pred):.4f}             ║')
print(f'║  Test F1 Score:    {f1_score(y_test, y_pred):.4f}             ║')
print(f'║  Test AUC:         {auc(*roc_curve(y_test, y_prob)[:2]):.4f}             ║')
print(f'║  CV AUC (5-fold):  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}     ║')
print('╚════════════════════════════════════════╝')
print(f'\nModel learned {X.shape[1]} weights + 1 bias (intercept = {clf_base.intercept_[0]:.4f})')

<a id='evaluation'></a>
## 6. 📊 Model Evaluation

We evaluate using multiple metrics as discussed in the lectures:
- **Confusion Matrix** — raw error counts
- **Precision / Recall / F1** — per-class performance
- **ROC Curve & AUC** — threshold-independent ranking quality
- **Precision-Recall Curve** — especially important for medical applications

In [ ]:
# Comprehensive evaluation dashboard
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# --- 1. Confusion Matrix ---
ax1 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Pred Malignant', 'Pred Benign'],
            yticklabels=['True Malignant', 'True Benign'],
            annot_kws={'size': 16})
ax1.set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# --- 2. Normalized Confusion Matrix ---
ax2 = fig.add_subplot(gs[0, 1])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax2,
            xticklabels=['Pred Malignant', 'Pred Benign'],
            yticklabels=['True Malignant', 'True Benign'],
            annot_kws={'size': 14}, vmin=0, vmax=1)
ax2.set_title('Normalized Confusion Matrix', fontsize=13, fontweight='bold')

# --- 3. Predicted Probability Distribution ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(y_prob[y_test == 0], bins=20, alpha=0.6, color=RED, label='Malignant', density=True)
ax3.hist(y_prob[y_test == 1], bins=20, alpha=0.6, color=BLUE, label='Benign', density=True)
ax3.axvline(0.5, color='k', ls='--', lw=2, label='Threshold = 0.5')
ax3.set_xlabel('P(Benign)', fontsize=11)
ax3.set_ylabel('Density', fontsize=11)
ax3.set_title('Predicted Probability Distribution', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)

# --- 4. ROC Curve ---
ax4 = fig.add_subplot(gs[1, 0])
fpr, tpr, roc_thresh = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax4.plot(fpr, tpr, c=BLUE, lw=2.5, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax4.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.5)')
ax4.fill_between(fpr, 0, tpr, alpha=0.1, color=BLUE)
ax4.set_xlabel('False Positive Rate', fontsize=11)
ax4.set_ylabel('True Positive Rate', fontsize=11)
ax4.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10, loc='lower right')

# --- 5. Precision-Recall Curve ---
ax5 = fig.add_subplot(gs[1, 1])
prec, rec, pr_thresh = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
ax5.plot(rec, prec, c=GREEN, lw=2.5, label=f'AP = {ap:.3f}')
ax5.fill_between(rec, 0, prec, alpha=0.1, color=GREEN)
ax5.axhline(y_test.mean(), color='gray', ls=':', lw=1, label=f'Baseline = {y_test.mean():.2f}')
ax5.set_xlabel('Recall', fontsize=11)
ax5.set_ylabel('Precision', fontsize=11)
ax5.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
ax5.legend(fontsize=10)
ax5.set_xlim(0, 1.02)
ax5.set_ylim(0, 1.05)

# --- 6. Metrics Summary ---
ax6 = fig.add_subplot(gs[1, 2])
tn, fp, fn, tp = cm.ravel()
metrics_names = ['Accuracy', 'Precision\n(Benign)', 'Recall\n(Benign)', 'F1 Score',
                 'Specificity\n(Malig. recall)', 'AUC']
metrics_vals = [
    (tp+tn)/(tp+tn+fp+fn),
    tp/(tp+fp),
    tp/(tp+fn),
    2*tp/(2*tp+fp+fn),
    tn/(tn+fp),
    roc_auc
]
colors_bar = [BLUE, GREEN, ORANGE, PURPLE, RED, BLUE]
bars = ax6.barh(metrics_names, metrics_vals, color=colors_bar, edgecolor='k', linewidth=0.5)
for bar, val in zip(bars, metrics_vals):
    ax6.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=12, fontweight='bold')
ax6.set_xlim(0, 1.12)
ax6.set_title('Performance Metrics', fontsize=13, fontweight='bold')
ax6.axvline(0.95, color='gray', ls=':', lw=1, alpha=0.5)

plt.suptitle('Logistic Regression — Breast Cancer Diagnostic Evaluation',
             fontsize=16, fontweight='bold', y=1.01)
plt.show()

# Classification report
print('\n' + '═'*60)
print('CLASSIFICATION REPORT')
print('═'*60)
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

<a id='threshold'></a>
## 7. 🎚️ Threshold Tuning for Clinical Use

In medical applications, the **default threshold of 0.5** is often not optimal.

### The Clinical Dilemma

| Error Type | Medical Meaning | Consequence |
|------------|-----------------|-------------|
| **False Negative** (FN) | Malignant tumour classified as benign | Patient misses treatment — **potentially fatal** |
| **False Positive** (FP) | Benign tumour classified as malignant | Unnecessary biopsy/anxiety — costly but not dangerous |

> In cancer screening, **missing a malignant case (FN) is far worse** than a false alarm (FP).  
> We should **lower the threshold** to classify more cases as malignant, increasing recall for malignant at the cost of more false positives.

**Note:** Since sklearn encodes Malignant=0 and Benign=1, lowering the threshold for "malignant" means *raising* the P(benign) threshold — i.e., we only call it benign if we are very confident.

In [ ]:
# Threshold analysis: how metrics change as we vary the threshold
thresholds = np.linspace(0.01, 0.99, 200)
results = []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_t).ravel()
    results.append({
        'threshold': t,
        'accuracy': (tp_t+tn_t)/(tp_t+tn_t+fp_t+fn_t),
        'precision_benign': tp_t/(tp_t+fp_t) if (tp_t+fp_t) > 0 else 0,
        'recall_benign': tp_t/(tp_t+fn_t) if (tp_t+fn_t) > 0 else 0,
        'recall_malignant': tn_t/(tn_t+fp_t) if (tn_t+fp_t) > 0 else 0,
        'f1': f1_score(y_test, y_t, zero_division=0),
        'fn': fn_t,  # missed malignant cases
        'fp': fp_t,  # false alarms
    })

df_thresh = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Left: Metrics vs threshold
ax = axes[0]
ax.plot(df_thresh['threshold'], df_thresh['accuracy'], lw=2, label='Accuracy', c=BLUE)
ax.plot(df_thresh['threshold'], df_thresh['recall_malignant'], lw=2.5,
        label='Malignant Recall (Specificity)', c=RED)
ax.plot(df_thresh['threshold'], df_thresh['recall_benign'], lw=2,
        label='Benign Recall (Sensitivity)', c=GREEN)
ax.plot(df_thresh['threshold'], df_thresh['f1'], lw=2, label='F1 Score', c=PURPLE, ls='--')
ax.axvline(0.5, color='gray', ls=':', lw=1.5, label='Default (0.5)')

# Find threshold that maximizes malignant recall while keeping accuracy > 0.95
safe = df_thresh[(df_thresh['recall_malignant'] >= 0.98) & (df_thresh['accuracy'] >= 0.90)]
if len(safe) > 0:
    best_t = safe.iloc[safe['f1'].argmax()]['threshold']
    ax.axvline(best_t, color=ORANGE, ls='--', lw=2, label=f'Clinical ({best_t:.2f})')

ax.set_xlabel('Classification Threshold (P(Benign))', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Metrics vs. Threshold', fontsize=14)
ax.legend(fontsize=9, loc='center left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

# Right: Error counts vs threshold
ax = axes[1]
ax.plot(df_thresh['threshold'], df_thresh['fn'], lw=2.5, c=RED, label='False Negatives (missed cancer!)')
ax.plot(df_thresh['threshold'], df_thresh['fp'], lw=2, c=ORANGE, label='False Positives (false alarms)')
ax.axvline(0.5, color='gray', ls=':', lw=1.5)
if len(safe) > 0:
    ax.axvline(best_t, color=ORANGE, ls='--', lw=2)
ax.set_xlabel('Classification Threshold (P(Benign))', fontsize=12)
ax.set_ylabel('Error Count', fontsize=12)
ax.set_title('Error Trade-off: FN (dangerous) vs FP (costly)', fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

if len(safe) > 0:
    row = safe.iloc[safe['f1'].argmax()]
    print(f'\n🏥 Recommended clinical threshold: {best_t:.2f}')
    print(f'   Malignant recall: {row["recall_malignant"]:.1%} (catches nearly all cancers)')
    print(f'   False negatives:  {int(row["fn"])} (missed cancers out of {(y_test==0).sum()} malignant cases)')
    print(f'   False positives:  {int(row["fp"])} (false alarms out of {(y_test==1).sum()} benign cases)')
    print(f'   Overall accuracy: {row["accuracy"]:.1%}')

<a id='regularization'></a>
## 8. 🛡️ Regularization Comparison

We compare **no regularization**, **L2 (Ridge)**, and **L1 (Lasso)** across a range of regularization strengths.

Recall:
- **L2** shrinks all weights toward zero — handles multicollinearity
- **L1** drives some weights exactly to zero — automatic feature selection

In [ ]:
# Regularization sweep: L1 vs L2 vs no regularization
C_values = np.logspace(-4, 4, 50)

results_reg = {'l1': {'C': [], 'auc': [], 'coefs': []},
               'l2': {'C': [], 'auc': [], 'coefs': []}}

for C_val in C_values:
    for penalty in ['l1', 'l2']:
        clf_r = LogisticRegression(C=C_val, penalty=penalty, solver='saga',
                                   max_iter=10000, random_state=SEED)
        clf_r.fit(X_train_sc, y_train)
        y_prob_r = clf_r.predict_proba(X_test_sc)[:, 1]
        auc_r = auc(*roc_curve(y_test, y_prob_r)[:2])

        results_reg[penalty]['C'].append(C_val)
        results_reg[penalty]['auc'].append(auc_r)
        results_reg[penalty]['coefs'].append(clf_r.coef_[0].copy())

# No regularization baseline
clf_none = LogisticRegression(penalty=None, max_iter=10000, random_state=SEED)
clf_none.fit(X_train_sc, y_train)
auc_none = auc(*roc_curve(y_test, clf_none.predict_proba(X_test_sc)[:, 1])[:2])

print('✅ Regularization sweep complete.')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: AUC vs regularization strength
ax = axes[0]
ax.plot(results_reg['l2']['C'], results_reg['l2']['auc'], c=BLUE, lw=2.5, label='L2 (Ridge)')
ax.plot(results_reg['l1']['C'], results_reg['l1']['auc'], c=ORANGE, lw=2.5, label='L1 (Lasso)')
ax.axhline(auc_none, color='gray', ls='--', lw=1.5, label=f'No regularization (AUC={auc_none:.3f})')
ax.set_xscale('log')
ax.set_xlabel('C  (inverse regularization strength → )', fontsize=11)
ax.set_ylabel('Test AUC', fontsize=11)
ax.set_title('AUC vs. Regularization Strength', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.9, 1.005)
ax.annotate('Stronger reg.', xy=(1e-3, 0.905), fontsize=9, color='gray')
ax.annotate('Weaker reg.', xy=(1e3, 0.905), fontsize=9, color='gray', ha='right')

# Panel 2: L2 regularization path
ax = axes[1]
coefs_l2 = np.array(results_reg['l2']['coefs'])
# Plot top 6 features by final magnitude
final_mag = np.abs(coefs_l2[-1])
top_idx = np.argsort(final_mag)[-6:]
for idx in top_idx:
    ax.plot(results_reg['l2']['C'], coefs_l2[:, idx], lw=1.5,
            label=data.feature_names[idx].replace('mean ', 'μ ').replace('worst ', 'w '))
ax.set_xscale('log')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('C (→ weaker regularization)', fontsize=11)
ax.set_ylabel('Coefficient value', fontsize=11)
ax.set_title('L2 Regularization Path (Top 6)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')

# Panel 3: L1 regularization path (shows sparsity)
ax = axes[2]
coefs_l1 = np.array(results_reg['l1']['coefs'])
# Count non-zero coefficients
n_nonzero = np.array([(np.abs(c) > 1e-6).sum() for c in coefs_l1])

ax2 = ax.twinx()
for idx in top_idx:
    ax.plot(results_reg['l1']['C'], coefs_l1[:, idx], lw=1.5, alpha=0.7,
            label=data.feature_names[idx].replace('mean ', 'μ ').replace('worst ', 'w '))
ax2.plot(results_reg['l1']['C'], n_nonzero, 'k--', lw=2, alpha=0.5, label='# non-zero')
ax.set_xscale('log')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('C (→ weaker regularization)', fontsize=11)
ax.set_ylabel('Coefficient value', fontsize=11)
ax2.set_ylabel('# Non-zero coefficients', fontsize=11, color='gray')
ax.set_title('L1 Regularization Path (Sparse!)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

# Find optimal C via cross-validation
best_auc_l2 = max(results_reg['l2']['auc'])
best_C_l2 = results_reg['l2']['C'][np.argmax(results_reg['l2']['auc'])]
best_auc_l1 = max(results_reg['l1']['auc'])
best_C_l1 = results_reg['l1']['C'][np.argmax(results_reg['l1']['auc'])]

# Count non-zero at best L1
best_l1_idx = np.argmax(results_reg['l1']['auc'])
n_active = (np.abs(results_reg['l1']['coefs'][best_l1_idx]) > 1e-6).sum()

print(f'📊 Best L2: C={best_C_l2:.4f}, AUC={best_auc_l2:.4f} (all {X.shape[1]} features used)')
print(f'📊 Best L1: C={best_C_l1:.4f}, AUC={best_auc_l1:.4f} (only {n_active}/{X.shape[1]} features ≠ 0)')
print(f'📊 No reg:  AUC={auc_none:.4f}')
print(f'\n📌 L1 achieves similar performance using fewer features — automatic feature selection!')

<a id='features'></a>
## 9. 🔬 Feature Importance & Clinical Interpretation

In logistic regression, the **magnitude and sign** of the learned weights $\boldsymbol{\theta}$ directly tell us which features drive the classification:

- **Positive** $\theta_j$: increasing feature $x_j$ makes the model predict **benign** (class 1)
- **Negative** $\theta_j$: increasing feature $x_j$ makes the model predict **malignant** (class 0)
- **Larger** $|\theta_j|$: stronger influence on the prediction

In [ ]:
# Feature importance comparison: L2 vs L1
clf_l2_best = LogisticRegression(C=best_C_l2, penalty='l2', solver='saga',
                                  max_iter=10000, random_state=SEED)
clf_l1_best = LogisticRegression(C=best_C_l1, penalty='l1', solver='saga',
                                  max_iter=10000, random_state=SEED)
clf_l2_best.fit(X_train_sc, y_train)
clf_l1_best.fit(X_train_sc, y_train)

# Sort by L2 magnitude
coefs_l2_best = clf_l2_best.coef_[0]
coefs_l1_best = clf_l1_best.coef_[0]
sorted_idx = np.argsort(np.abs(coefs_l2_best))

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# Left: L2 weights
ax = axes[0]
colors_w = [GREEN if c > 0 else RED for c in coefs_l2_best[sorted_idx]]
ax.barh(range(30), coefs_l2_best[sorted_idx], color=colors_w, edgecolor='k', linewidth=0.3)
ax.set_yticks(range(30))
ax.set_yticklabels([data.feature_names[i] for i in sorted_idx], fontsize=8)
ax.set_xlabel('Coefficient (L2)', fontsize=11)
ax.set_title('L2 Ridge — All 30 Features Active', fontsize=13, fontweight='bold')
ax.axvline(0, color='k', lw=0.8)

# Right: L1 weights (showing sparsity)
ax = axes[1]
colors_w = [GREEN if c > 0 else (RED if c < 0 else 'lightgray') for c in coefs_l1_best[sorted_idx]]
ax.barh(range(30), coefs_l1_best[sorted_idx], color=colors_w, edgecolor='k', linewidth=0.3)
ax.set_yticks(range(30))
ax.set_yticklabels([data.feature_names[i] for i in sorted_idx], fontsize=8)
ax.set_xlabel('Coefficient (L1)', fontsize=11)
n_zero = (np.abs(coefs_l1_best) < 1e-6).sum()
ax.set_title(f'L1 Lasso — {30 - n_zero} Features Active, {n_zero} Eliminated', fontsize=13, fontweight='bold')
ax.axvline(0, color='k', lw=0.8)

# Color legend
fig.text(0.5, -0.01,
         '🟢 Green = higher value pushes toward Benign   │   🔴 Red = higher value pushes toward Malignant   │   ⬜ Gray = eliminated by L1',
         ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('Feature Importance: Which Cell Measurements Drive the Diagnosis?',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Clinical interpretation of top features
print('═' * 70)
print('CLINICAL INTERPRETATION — Top Features Predicting Malignancy')
print('═' * 70)

# Top features pushing toward malignant (negative coefficients)
mal_idx = np.argsort(coefs_l2_best)[:5]  # most negative
ben_idx = np.argsort(coefs_l2_best)[-5:][::-1]  # most positive

print('\n🔴 Top features associated with MALIGNANT tumours:')
print('   (Higher values → more likely malignant)\n')
for i, idx in enumerate(mal_idx, 1):
    print(f'   {i}. {data.feature_names[idx]:<30s}  θ = {coefs_l2_best[idx]:+.3f}')

print('\n🟢 Top features associated with BENIGN tumours:')
print('   (Higher values → more likely benign)\n')
for i, idx in enumerate(ben_idx, 1):
    print(f'   {i}. {data.feature_names[idx]:<30s}  θ = {coefs_l2_best[idx]:+.3f}')

print('\n' + '─' * 70)
print('📌 Key clinical insight: Worst concave points, worst perimeter, and')
print('   mean concavity are the strongest indicators of malignancy.')
print('   This aligns with medical knowledge — malignant cells have more')
print('   irregular shapes with deeper indentations (concave features).')

<a id='pca'></a>
## 10. 🗺️ PCA-Projected Decision Boundary

With 30 features, we cannot directly plot the decision boundary. But by projecting onto the first two **principal components**, we can visualize how the model separates the classes in the directions of maximum variance.

In [ ]:
# PCA projection for visualization
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca = pca.transform(X_test_sc)

print(f'PCA explained variance: PC1 = {pca.explained_variance_ratio_[0]:.1%}, '
      f'PC2 = {pca.explained_variance_ratio_[1]:.1%}, '
      f'Total = {pca.explained_variance_ratio_.sum():.1%}')

# Fit logistic regression on PCA features for boundary visualization
clf_pca = LogisticRegression(C=1.0, max_iter=5000, random_state=SEED)
clf_pca.fit(X_train_pca, y_train)
acc_pca = accuracy_score(y_test, clf_pca.predict(X_test_pca))
auc_pca = auc(*roc_curve(y_test, clf_pca.predict_proba(X_test_pca)[:, 1])[:2])

# Create decision boundary grid
xx, yy = np.meshgrid(
    np.linspace(X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1, 300),
    np.linspace(X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = clf_pca.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Left: Decision boundary on training data
ax = axes[0]
im = ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 50), cmap='RdBu', alpha=0.4)
ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2.5, linestyles='--')

ax.scatter(X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1],
           c=RED, s=25, alpha=0.6, edgecolors='k', linewidths=0.2, label='Malignant')
ax.scatter(X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1],
           c=BLUE, s=25, alpha=0.6, edgecolors='k', linewidths=0.2, label='Benign')

# Draw θ vector
theta_pca = clf_pca.coef_[0]
scale = 3.0 / np.linalg.norm(theta_pca)
ax.annotate('', xy=(theta_pca[0]*scale, theta_pca[1]*scale),
            xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=ORANGE, lw=3))
ax.text(theta_pca[0]*scale + 0.3, theta_pca[1]*scale + 0.3,
        '$\\boldsymbol{\\theta}$', fontsize=14, color=ORANGE, fontweight='bold')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title('Training Data — Decision Boundary in PCA Space', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Right: Test data with probability heatmap
ax = axes[1]
im = ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 50), cmap='RdBu', alpha=0.4)
ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2.5, linestyles='--')

# Color test points by correct/incorrect
y_pred_pca = clf_pca.predict(X_test_pca)
correct = y_pred_pca == y_test.values

ax.scatter(X_test_pca[correct & (y_test.values == 0), 0], X_test_pca[correct & (y_test.values == 0), 1],
           c=RED, s=40, alpha=0.8, edgecolors='k', linewidths=0.3, marker='o', label='Malig. (correct)')
ax.scatter(X_test_pca[correct & (y_test.values == 1), 0], X_test_pca[correct & (y_test.values == 1), 1],
           c=BLUE, s=40, alpha=0.8, edgecolors='k', linewidths=0.3, marker='o', label='Benign (correct)')
ax.scatter(X_test_pca[~correct, 0], X_test_pca[~correct, 1],
           c='yellow', s=100, alpha=1.0, edgecolors='k', linewidths=1.5, marker='X',
           label=f'Misclassified ({(~correct).sum()})', zorder=5)

plt.colorbar(im, ax=ax, label='P(Benign)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title(f'Test Data — Acc: {acc_pca:.1%}, AUC: {auc_pca:.3f} (2 PCs only)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower left')

plt.tight_layout()
plt.show()

print(f'\n📌 Note: Using only 2 PCs ({pca.explained_variance_ratio_.sum():.1%} of variance), '
      f'accuracy = {acc_pca:.1%}.')
print(f'   The full 30-feature model achieves {accuracy_score(y_test, y_pred):.1%} — '
      f'the remaining PCs carry additional discriminating information.')

In [ ]:
# What do the PCA loadings tell us?
loadings = pd.DataFrame(pca.components_.T, index=data.feature_names,
                         columns=['PC1', 'PC2'])

fig, ax = plt.subplots(figsize=(8, 6))

# Plot loading vectors
for i, feat in enumerate(data.feature_names):
    ax.annotate('', xy=(loadings.iloc[i, 0], loadings.iloc[i, 1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=BLUE, lw=0.8, alpha=0.5))

# Highlight top features
top_load = loadings['PC1'].abs().nlargest(5).index
for feat in top_load:
    row = loadings.loc[feat]
    ax.annotate('', xy=(row['PC1'], row['PC2']), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=RED, lw=2))
    ax.text(row['PC1'] * 1.05, row['PC2'] * 1.05,
            feat.replace('mean ', 'μ ').replace('worst ', 'w '),
            fontsize=8, fontweight='bold', color=RED)

circle = plt.Circle((0, 0), 1, color='gray', fill=False, ls='--', lw=0.5)
ax.add_patch(circle)
ax.axhline(0, color='k', lw=0.3)
ax.axvline(0, color='k', lw=0.3)
ax.set_xlabel(f'PC1 Loading ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 Loading ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title('PCA Loading Plot — Which Features Drive Each Component?', fontsize=13, fontweight='bold')
ax.set_aspect('equal')
ax.set_xlim(-0.35, 0.35)
ax.set_ylim(-0.55, 0.55)
plt.tight_layout()
plt.show()

print('📌 PC1 is dominated by size/shape features (radius, perimeter, area, concavity).')
print('   This single component captures the major axis of variation between benign & malignant.')

<a id='summary'></a>
## 11. 📝 Summary & Clinical Insights

### Results Overview

In [ ]:
# Final summary table
print('╔' + '═'*68 + '╗')
print('║' + '  BREAST CANCER DIAGNOSIS — LOGISTIC REGRESSION RESULTS'.center(68) + '║')
print('╠' + '═'*68 + '╣')

# Compare models
models = [
    ('No Regularization', clf_none),
    (f'L2 Ridge (C={best_C_l2:.3f})', clf_l2_best),
    (f'L1 Lasso (C={best_C_l1:.3f})', clf_l1_best),
    ('2-PC Model (PCA)', clf_pca),
]

print('║' + f'{"Model":<30s} {"Acc":>7s} {"AUC":>7s} {"F1":>7s} {"# Feat":>8s}' + '  ║')
print('║' + '─'*66 + '  ║')

for name, clf in models:
    if name.startswith('2-PC'):
        X_te_m = X_test_pca
        n_feat = '2 PCs'
    else:
        X_te_m = X_test_sc
        n_feat_count = (np.abs(clf.coef_[0]) > 1e-6).sum()
        n_feat = str(n_feat_count)

    y_p = clf.predict(X_te_m)
    y_pr = clf.predict_proba(X_te_m)[:, 1]
    acc_m = accuracy_score(y_test, y_p)
    auc_m = auc(*roc_curve(y_test, y_pr)[:2])
    f1_m = f1_score(y_test, y_p)
    print('║' + f'  {name:<28s} {acc_m:>6.3f} {auc_m:>7.3f} {f1_m:>7.3f} {n_feat:>7s}' + '  ║')

print('╚' + '═'*68 + '╝')

### Key Takeaways

1. **Logistic regression achieves excellent performance** on this diagnostic task (~97% accuracy, ~99% AUC) — a linear classifier suffices when features are well designed.

2. **Feature scaling is essential** — the raw features span orders of magnitude (area ~100s vs smoothness ~0.1). Without standardization, gradient descent converges poorly.

3. **Multicollinearity is present** — radius, perimeter, and area are highly correlated (r > 0.95). L2 regularization distributes weight among correlated features, stabilizing the solution.

4. **L1 regularization performs automatic feature selection** — it achieves comparable AUC while zeroing out many of the 30 features, identifying the most informative subset.

5. **Threshold tuning matters in medical applications** — the cost of a false negative (missed cancer) is much higher than a false positive (unnecessary biopsy). Lowering the benign threshold reduces missed cancers.

6. **Model interpretability is a major advantage** — we can directly read which cell features (concavity, perimeter, radius) drive the diagnosis, aligning with clinical knowledge.

7. **PCA reveals that 2 components capture most of the diagnostic signal** — the first PC alone, driven by size/shape features, provides strong class separation.

---

### 🔄 Pipeline Recap

$$
\text{Raw FNA Data} \;\xrightarrow{\text{Standardize}}\; \mathbf{X} \;\xrightarrow{\text{Logistic Regression}}\; P(\text{benign}\mid\boldsymbol{x}) \;\xrightarrow{\text{Threshold}}\; \text{Diagnosis}
$$

> **Logistic regression is a powerful, interpretable, and well-calibrated baseline.**
> In many medical applications, its transparency (coefficients as risk factors) makes it preferred over black-box alternatives.

---

*Notebook by **Adil Rasheed** — Department of Engineering Cybernetics, NTNU*  
*TTK4260: Multivariate Data Analysis & Machine Learning — Spring 2026*